# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\chapter06
프로젝트 루트: c:\dev\llm-data-analysis-study\chapter06
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\chapter06\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\chapter06\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [3]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 0
주문 시작일: 2025-09-18 00:00:00
주문 종료일: 2026-09-17 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [5]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [6]:
customers['age'].describe()


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [7]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


In [8]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,F,84,56.0
1,M,66,44.0


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [9]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


In [10]:
products['price'].describe()


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [11]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [12]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


In [13]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


In [14]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-09-18 00:00:00
주문 종료일: 2026-09-17 00:00:00
주문 월 개수: 13


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [15]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [16]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (764, 6)
병합 후 sales_items: (764, 9)
상품명 누락: 0
카테고리 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,100,3,102000,306000,도서 상품 100,도서,102000
1,2,1,87,5,25000,125000,도서 상품 087,도서,25000
2,3,1,7,3,142000,426000,도서 상품 007,도서,142000
3,4,1,9,3,193000,579000,스포츠 상품 009,스포츠,193000
4,5,2,72,4,189000,756000,뷰티 상품 072,뷰티,189000


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [17]:
sales_items = sales_items.merge(orders[['order_id', 'order_status']], on='order_id', how='left')
sales_items = sales_items[sales_items['order_status'] == 'completed']

In [18]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


In [19]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,295,31743000,21.31,111579.0
1,전자기기,259,26400000,17.72,101588.0
2,생활용품,272,23915000,16.05,96438.0
3,뷰티,223,23383000,15.69,117688.0
4,식품,133,16573000,11.12,137143.0
5,도서,149,16389000,11.00,106857.0
6,패션,111,10587000,7.11,115909.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [20]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (764, 12)
order_date 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,100,3,102000,306000,123,2026-07-17,card,completed,2026-07,Friday
1,2,1,87,5,25000,125000,123,2026-07-17,card,completed,2026-07,Friday
2,3,1,7,3,142000,426000,123,2026-07-17,card,completed,2026-07,Friday
3,4,1,9,3,193000,579000,123,2026-07-17,card,completed,2026-07,Friday
4,5,2,72,4,189000,756000,77,2025-10-02,naver_pay,cancelled,2025-10,Thursday


In [21]:

monthly_sales = (
    order_sales[order_sales['order_status'] == 'completed']
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,14566000,16,910375.0
2,2025-11,11403000,14,814500.0
3,2025-12,20963000,22,952864.0
4,2026-01,16416000,19,864000.0
5,2026-02,3861000,6,643500.0
6,2026-03,19010000,25,760400.0
7,2026-04,10490000,16,655625.0
8,2026-05,14900000,18,827778.0
9,2026-06,14244000,17,837882.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [22]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [23]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [24]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('c:/dev/llm-data-analysis-study/chapter06/reports/ch06_category_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter06/reports/ch06_customer_city.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter06/reports/ch06_customer_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter06/reports/ch06_eda_questions.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter06/reports/ch06_monthly_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/chapter06/reports/ch06_product_category.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [25]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: c:\dev\llm-data-analysis-study\chapter06\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [29]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [30]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [36]:
full = (order_items
    .merge(orders, on='order_id', how='left')
    .merge(products[['product_id', 'category']], on='product_id', how='left')
    .merge(customers[['customer_id', 'city']], on='customer_id', how='left'))
full['order_date'] = pd.to_datetime(full['order_date'], errors='coerce')
full['order_month'] = full['order_date'].dt.to_period('M').astype(str)

done = full[full['order_status'] == 'completed']
print(done['line_total'].sum())   # 148990000이어야 함

148990000


In [37]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
city_customers = customers.groupby('city', as_index=False).agg(customer_count=('customer_id', 'nunique'))
city_sales = done.groupby('city', as_index=False).agg(total_sales=('line_total', 'sum'))

city_summary = city_customers.merge(city_sales, on='city', how='left').fillna({'total_sales': 0})
city_summary['sales_per_customer'] = (city_summary['total_sales'] / city_summary['customer_count']).round(0)
city_summary.sort_values('total_sales', ascending=False)

,city,customer_count,total_sales,sales_per_customer
6,성남,21,24099000,1147571.0
0,고양,11,19116000,1737818.0
5,서울,15,19097000,1273133.0
8,울산,14,15380000,1098571.0
1,광주,17,14264000,839059.0
4,부산,16,13604000,850250.0
9,인천,14,12832000,916571.0
3,대전,14,11792000,842286.0
7,수원,13,10620000,816923.0
2,대구,15,8186000,545733.0


In [38]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
product_count = products.groupby('category', as_index=False).agg(product_count=('product_id', 'nunique'))
cat_sales = done.groupby('category', as_index=False).agg(total_sales=('line_total', 'sum'), total_quantity=('quantity', 'sum'))

cat_compare = product_count.merge(cat_sales, on='category')
cat_compare['product_rank'] = cat_compare['product_count'].rank(ascending=False, method='min')
cat_compare['sales_rank'] = cat_compare['total_sales'].rank(ascending=False, method='min')
cat_compare['sales_per_product'] = (cat_compare['total_sales'] / cat_compare['product_count']).round(0)
cat_compare.sort_values('product_count', ascending=False)

,category,product_count,total_sales,total_quantity,product_rank,sales_rank,sales_per_product
3,스포츠,19,31743000,295,1.0,1.0,1670684.0
5,전자기기,17,26400000,259,2.0,2.0,1552941.0
1,뷰티,16,23383000,223,3.0,4.0,1461438.0
2,생활용품,16,23915000,272,3.0,3.0,1494688.0
0,도서,14,16389000,149,5.0,6.0,1170643.0
6,패션,11,10587000,111,6.0,7.0,962455.0
4,식품,7,16573000,133,7.0,5.0,2367571.0


In [39]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
order_level = (done.groupby(['order_id', 'payment_method'], as_index=False)
    .agg(order_total=('line_total', 'sum')))

payment_avg = (order_level.groupby('payment_method', as_index=False)
    .agg(order_count=('order_id', 'nunique'),
         total_sales=('order_total', 'sum'),
         avg_order_value=('order_total', 'mean'))
    .round(0)
    .sort_values('avg_order_value', ascending=False))
payment_avg

,payment_method,order_count,total_sales,avg_order_value
3,naver_pay,51,43500000,852941.0
1,card,39,31806000,815538.0
2,kakao_pay,49,39342000,802898.0
0,bank_transfer,45,34342000,763156.0


In [40]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
month_category = done.pivot_table(
    index='order_month', columns='category',
    values='line_total', aggfunc='sum', fill_value=0)
month_category['합계'] = month_category.sum(axis=1)
month_category.loc['합계'] = month_category.sum()
month_category

category,도서,뷰티,생활용품,스포츠,식품,전자기기,패션,합계
order_month,,,,,,,,
2025-09,410000,580000,893000,269000,0,498000,0,2650000
2025-10,602000,2764000,1162000,2641000,1890000,3897000,1610000,14566000
2025-11,1016000,1570000,1385000,2569000,1013000,2953000,897000,11403000
2025-12,1831000,3557000,4825000,5831000,394000,3594000,931000,20963000
2026-01,1589000,2786000,2509000,2960000,2776000,3232000,564000,16416000
2026-02,172000,301000,259000,489000,1933000,583000,124000,3861000
2026-03,3367000,2176000,2318000,5104000,1895000,3615000,535000,19010000
2026-04,1536000,2246000,1193000,2492000,385000,1074000,1564000,10490000
2026-05,1286000,1944000,1561000,3921000,2965000,1317000,1906000,14900000


# 과제 5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.

고객별로 order_count(고객의 주문 횟수)와 avg_order_value(고객의 주문 1건당 평균 금액) 두 지표를 함께 본다.

| 구분 | order_count | avg_order_value |
|---|---|---|
| 반복 구매 고객 | 2건 이상 | 제한 없음 |
| 고액 단발 고객 | 1건 | 전체 고객 중앙값 이상 |
| 일반 단발 고객 | 1건 | 전체 고객 중앙값 미만 |


# 과제 6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.
아래는 주문 데이터 분석에 쓸 수 있는 테이블과 컬럼 목록이다.
- customers: customer_id, city (이름 등 개인정보는 제외)
- orders: order_id, customer_id, order_date, payment_method, order_status (completed, cancelled, refunded)
- order_items: order_id, product_id, quantity, unit_price, line_total
- products: product_id, category, price
아래의 질문들을 하려고 하는데, 현재 데이터만으로 답할 수 없는 질문이면 이유를 한 줄로 설명하고, 현재 컬럼으로 답할 수 있는 질문으로 바꿔 줘.
질문: "왜 2026-08 매출이 낮았나?", "어떤 광고가 매출을 올렸나?", "고객이 왜 환불했나?"

바꾼 질문마다 아래 형식으로 답해 줘.
1. 바뀐 질문
2. 필요한 컬럼과 집계 방법(예: 월 × order_status 피벗)
3. 이 데이터로 확인할 수 있는 것과 확인할 수 없는 것

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

> 이 내용을 `chapter06.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 황지원
- GitHub ID: Jiwon0712
- 작성일: 2026-10-18
- 최종 Notebook URL: https://github.com/Jiwon0712/llm-data-analysis-study/blob/main//chapter06/chapter06.ipynb

## 1. 내가 정한 EDA 질문
| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 월별 매출은 어떻게 변하는가? | 전체 기간, 월 단위 | 월별 총매출 | `groupby()`로 월별 합계 계산 후 선 그래프 확인 |
| 카테고리별 매출 비중은 어떻게 다른가? | 전체 카테고리 | 카테고리별 매출 비중(%) | `groupby()`로 카테고리별 총매출 계산 후 전체 대비 비율 산출 |
| 매출 비중이 가장 큰 카테고리의 단가는 다른 카테고리보다 높은가? | 상위 카테고리 vs 나머지 | 평균 단가, 판매 수량 | `groupby()`로 카테고리별 평균 단가와 수량 비교 |

### 왜 이 질문을 선택했는가?
질문 1: 분석의 출발점, 매출이 시간에 따라 증가/감소하는지, 특정 시기에 몰리는 패턴이 있는지 먼저 파악하고자 하였다. 
질문 2: 매출이 어떤 상품군에 집중되어 있는지 알아야 어디를 더 자세히 봐야 할지 판단할 수 있다. 그룹 간 차이를 확인하는 가장 기본적인 질문.
질문 3: 카테고리별 비교에서 나온 관찰을 가설로 이어가기 위하여 선택하였다. 특정 카테고리의 매출 비중이 높게 나올 것이라는 생각을 염두에 두고, 그 이유가 단가가 높아서일 수 있다는 가설을 세우기 위함.

## 2. 핵심 집계 결과
- completed 원본 합계: 148990000

- 카테고리별 합계:

| category | total_sales |
|---|---:|
| 스포츠 | 31743000 |
| 전자기기 | 26400000 |
| 생활용품 | 23915000 |
| 뷰티 | 23383000 |
| 식품 | 16573000 |
| 도서 | 16389000 |
| 패션 | 10587000 |
| **합계** | **148990000** |

- 월별 합계:

| order_month | total_sales |
|---|---:|
| 2025-09 | 2650000 |
| 2025-10 | 14566000 |
| 2025-11 | 11403000 |
| 2025-12 | 20963000 |
| 2026-01 | 16416000 |
| 2026-02 | 3861000 |
| 2026-03 | 19010000 |
| 2026-04 | 10490000 |
| 2026-05 | 14900000 |
| 2026-06 | 14244000 |
| 2026-07 | 11333000 |
| 2026-08 | 5103000 |
| 2026-09 | 4051000 |
| **합계** | **148990000** |

- 총합 일치 여부: 카테고리별/월별 전체 합계 모두 148,990,000으로 completed 원본 합계와 일치.

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
카테고리별 매출 합계는 스포츠, 전자기기, 생활용품, 뷰티, 식품, 도서, 패션 순으로 높다.
월별 매출 합계는 가장 높은 달은 2025-12, 가장 낮은 달은 2025-09이다. 
카테고리별/월별 매출의 전체 합계는 completed 원본 합계와 일치하므로 데이터에 누락은 없다. 

### 나의 해석과 판단
스포츠 한 카테고리가 전체의 약 21.3%를 차지하고, 상위 3개 카테고리를 합치면 약 54.7%다. 반면 패션은 약 7.1%로 가장 작아서 카테고리 간 편차가 크다.
월별로는 2025-12(20963000)와 2026-03(19010000)에서 매출이 높고, 2026-02(3861000)는 크게 낮다. 2025-09(2650000)와 2026-09(4051000)는 첫 달과 마지막 달이라 월 전체가 집계되지 않았을 가능성이 있어, 낮은 값을 실제 매출 부진으로 단정하기 어렵다.
이번 집계는 주문 상태가 completed인 건만 포함하므로 확정 매출 기준의 결과다.

### 업무·분석적 의미
- 매출 상위 카테고리인 스포츠와 전자기기는 재고 확보와 프로모션 우선순위를 정할 때 기준이 될 수 있다.
- 매출 비중이 낮은 패션, 도서는 판매 부진 원인을 따로 점검할 필요가 있다.
- 연말인 2025-12에 매출이 높았으므로 시즌성 수요에 대비한 재고/마케팅 계획을 검토할 수 있다.
- 필터 적용 전 전체 주문 합계(255610000)와 completed 합계(148990000)의 차이가 크므로, 매출 보고 시 기준을 명확히 구분해야 한다.

### 한계와 추가 확인 사항
- 이번 결과는 매출 금액만 본 것이므로 주문 건수, 수량, 평균 단가를 함께 봐야 매출 차이의 원인을 알 수 있다.
- 2025-09와 2026-09는 월 일부만 집계됐을 수 있어 기간의 시작일과 종료일 확인이 필요하다.
- completed가 아닌 주문이 전체 주문 금액의 약 41.7%에 달하므로, 상태별 비중과 원인을 추가로 확인해야 한다.
- 월별 추이는 13개월 데이터라 계절성이나 추세를 단정하기에는 기간이 짧다.

## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: completed 기준 카테고리 매출은 스포츠 31743000(21.3%)이 가장 크고 패션 10587000(7.1%)이 가장 작아 약 3.0배 차이가 난다. 수량 순위(스포츠 295, 생활용품 272, 전자기기 259, 뷰티 223, 도서 149, 식품 133, 패션 111)와 매출 순위는 대체로 비슷하지만 두 곳이 어긋난다. 생활용품은 수량 2위인데 매출은 3위이고(판매단가 약 87923으로 가장 낮음), 식품은 수량이 도서보다 적은데 매출은 더 높다(판매단가 약 124609 vs 109993). 상품 수는 스포츠 19개, 패션 11개, 식품 7개다.
- 가설: 카테고리 간 매출 차이는 주로 판매 수량에서 나오고, 단가는 순위를 일부 바꾸는 보조 요인이다. 패션과 도서는 상품당 판매 수량이 약 10개로 낮아 매출이 작고, 식품은 상품이 7개뿐이어도 상품당 19개가 팔리고 단가가 높아 매출 5위를 유지한다.
- 추가 검증: category_sales에 판매단가(total_sales ÷ total_quantity)와 상품당 판매 수량(total_quantity ÷ product_count)을 계산해 순위를 비교한다. 매출 차이를 수량 효과와 단가 효과로 나눠서 어느 쪽이 더 큰지 확인한다. 카테고리 상품 목록의 avg_price는 판매 가중이 아니므로 판매단가와 구분해서 해석한다.

### 사례 2
- 관찰: 2026-08은 전체 주문 20건 중 completed가 5건(25.0%)뿐이라 completed 매출이 5103000으로 낮다. 다른 달의 completed 주문 비율은 42.9%(2026-02)~73.3%(2025-12) 범위다. 2026-08 주문 건수 20건은 2026-07(23건)과 크게 다르지 않고, completed 5건의 평균 주문 금액은 1020600으로 13개월 중 가장 높다. 전체 기준으로 completed가 아닌 주문은 취소 64건, 환불 52건으로 116건(38.7%)이며, 이 주문들의 평균 금액은 약 919138로 completed(약 809728)보다 높다.
- 가설: 2026-08의 매출 하락은 수요 감소가 아니라 취소/환불이 그 달에 집중되어 생긴 결과다. 상태 값이 completed, cancelled, refunded 세 가지뿐이므로 처리 지연보다는 취소/환불 증가가 원인일 가능성이 크다. 취소/환불 주문의 평균 금액이 더 높은 것으로 보아 고액 주문일수록 취소/환불로 이어지는 경향도 있을 것이다.
- 추가 검증: 월 × order_status 피벗으로 2026-08의 cancelled와 refunded 건수를 확인한다. 같은 방식으로 14건 중 6건이 completed인 2026-02도 비교한다. 2026-08의 취소/환불 주문을 payment_method, category, customer_id로 나눠 한쪽에 몰려 있는지 본다. 월 주문 수가 5~38건으로 적어 변동이 클 수 있으므로 월별 completed 비율 평균과 함께 판단한다.

### 사례 3
- 관찰: 구매 금액 상위 10명(customer_id 102, 117, 83, 142, 40, 30, 3, 14, 146, 111)의 총매출은 4067000~5977000이다. 상위 10명 안에서도 유형이 갈린다. 5명(102, 117, 83, 40, 30)은 주문 5건 이상이고, 나머지 5명(142, 3, 14, 146, 111)은 주문이 3~4건인데 건당 평균이 1016750~1620667로 높다. customer_id 30(7건, 평균 693286)과 142(3건, 평균 1620667)는 총액이 4853000과 4862000으로 거의 같다. 도시는 서울과 성남이 각 3명이다.
- 가설: 고액 고객은 한 유형이 아니라 반복 구매형과 고단가형으로 나뉜다. 반복 구매가 고액의 주된 이유라면 상위 고객의 주문 횟수가 전체 고객 평균보다 뚜렷하게 높을 것이고, 고단가 구매가 이유라면 건당 금액이 전체 평균 주문 금액보다 높을 것이다. 상위 10명 중 9명이 후자 조건을 만족하므로 건당 금액의 영향이 더 커 보인다.
- 추가 검증: 고객별 주문 횟수 분포를 구해 상위 고객과 전체 고객의 평균 주문 횟수를 비교한다. 상위 10% 고객이 전체 매출에서 차지하는 누적 비중을 계산한다. 반복형과 고단가형을 나눠 선호 카테고리와 도시를 비교하고, 도시별 구매 금액 차이도 확인한다. 고객 집계표가 completed 기준인지 확인하고, 아니라면 필터 후 다시 계산한다.

## 4. LLM 질문 확장
- Safe Context: 원본 주문 행은 제공하지 않고 집계 결과표만 제공함(카테고리별 매출, 월별 매출, 주문 상태 분포, 결제수단 분포, 상위 고객 표). 컬럼 구조는 order_items, orders, products, customers 기준. 결과물에는 고객 이름을 쓰지 않고 customer_id만 사용함.

- Prompt: completed 기준 카테고리별/월별 집계 결과를 바탕으로 관찰 → 가설 → 추가 검증 사례를 작성하고 숫자가 맞는지 먼저 검증해 줘. 그다음 더 좋은 가설을 쓰기 위해 필요한 데이터와 확인 항목을 정리해 줘. 이후 상태 분포와 월별/고객 집계 결과를 줄 테니 그 숫자로 가설을 다시 써 줘.

- LLM 제안 요약: 가설 근거를 보강할 자료로 주문 상태 분포, 카테고리 × 상태 피벗, 월 × 상태 피벗, completed 기준 수량/판매단가, 날짜 범위, 이상치/집중도, 비교 지표(월별 completed 비율 등)를 제안함. 실제 데이터를 받은 뒤에는 사례 1(수량 vs 단가), 사례 2(2026-08 취소/환불 집중), 사례 3(고액 고객의 반복 구매형 vs 고단가형) 가설과 검증 방법을 제시함.

- 실제 사용 여부: 수정 후 사용

- 판단 이유: LLM의 첫 가설은 배송 중, 처리 중 주문이 많다는 가정을 포함했는데, 실제 주문 상태는 completed, cancelled, refunded 세 가지뿐이라 데이터와 맞지 않아 폐기하고 취소/환불 중심으로 바꿈. 상위 3개 카테고리 비중이 54.7%로 잘못 계산되어 있어 직접 다시 계산해 55.1%로 고침. 가설은 필터 전 대비 completed 비율 같은 숫자에 근거해 사실로 단정하지 않고 검증 필요 항목으로만 남김.

- 사람이 수정한 내용: 원본 코드는 고치지 않고 completed 필터만 추가하도록 요청해 sales_items와 order_sales에 필터를 적용함. 필터 적용 후 카테고리별·월별 합계가 148990000으로 completed 원본 합계와 일치하는지 확인함. 숫자에서 콤마를 제거하고 형식을 노트북에 맞춤. 필터 전 값(255610000)으로 쓰인 수량, 단가를 completed 기준으로 다시 계산하도록 고침.

![LLM EDA 질문 확장](images/step04_llm.png)

## 5. 다음 분석 우선순위
1. 월 × order_status 피벗과 카테고리 × order_status 피벗 (cancelled, refunded가 특정 월, 카테고리에 몰려 있는지 확인)
2. completed 기준 카테고리별 판매단가(total_sales ÷ total_quantity)와 상품당 판매 수량 비교. 즉 매출 차이가 수량 때문인지 단가 때문인지 분해.
3. 고객별 주문 횟수 분포와 상위 고객 매출 누적 비중. 즉 고액 고객이 반복 구매형인지 고단가형인지 확인.

### 가장 먼저 확인하고 싶은 이유
completed 매출은 148990000으로, 전체 주문 금액 255610000의 약 58.3%뿐이다. 나머지 약 41.7%가 취소·환불이라 매출 해석이 어떤 상태가 어디에 몰렸는지에 크게 좌우된다. 주문 20건 중 completed 주문이 5건인 2026-08처럼 월별 매출이 급격히 낮은 구간도 수요 감소인지 취소/환불 집중인지 상태 피벗으로 먼저 구분해야 이후 카테고리, 고객 분석의 기준이 흔들리지 않는다.

### 현재 데이터만으로 단정할 수 없는 것
- 2026-08 등 특정 월의 낮은 completed 매출이 취소/환불 집중 때문인지는 상태별 월 분포 없이는 확정할 수 없다.
- 카테고리 간 매출 차이가 수량 때문인지 단가 때문인지, 현재는 순위 비교로 추정만 한 상태다.
- 고액 고객이 반복 구매형인지 고단가형인지, 고객 표가 completed 기준인지 확인되지 않았다.
- 2025-09, 2026-09는 주문이 각 5건뿐이라 부분 월인지 확정할 수 없다(order_date의 min, max 확인 필요).
- 13개월 데이터라 계절성이나 추세를 단정하기에는 기간이 짧다.
- 취소/환불의 원인은 데이터에 없어 알 수 없다.

## 최종 체크
- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [x] 최종 Notebook URL을 제출합니다.